# 04 · Logistic Regression (Bonus Model)
**Payment Failure Intelligence & Revenue Optimization System**

**Goal:** Build a predictive model to estimate the probability of failure 
for a transaction based on time, amount, category, and device.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Style for plots
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444', 'text.color': '#e0e0e0'
})

## 1. Load Data
We'll use the engineered dataset which already has our target variable `is_failed`
and useful predictors like `hour`, `amount`, `category`, and `device_type`.

In [ ]:
print("Loading data...")
df = pd.read_csv('../outputs/engineered_data.csv')
print(f"Data shape: {df.shape}")

# Features and target
X = df[['hour', 'amount', 'category', 'device_type', 'is_night']]
y = df['is_failed']

## 2. Preprocessing & Pipeline Setup
We need to scale numeric features and one-hot encode categorical features.

In [ ]:
numeric_features = ['hour', 'amount', 'is_night']
numeric_transformer = StandardScaler()

categorical_features = ['category', 'device_type']
categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop='first')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create the pipeline with a Logistic Regression model
# We use class_weight='balanced' because failures are a minority class (15%)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

## 3. Train-Test Split & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set: {X_test.shape[0]} rows")

print("Training model...")
model.fit(X_train, y_train)
print("Training complete.")

## 4. Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Classification Report
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# ROC AUC
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC AUC Score: {roc_auc:.4f}")

## 5. Confusion Matrix & ROC Curve Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0], 
            cbar=False, annot_kws={'size': 14})
axes[0].set_title('Confusion Matrix', fontsize=14, color='white')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_xticklabels(['Success (0)', 'Failed (1)'])
axes[0].set_yticklabels(['Success (0)', 'Failed (1)'])

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#ff6584', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='#aaa', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=14, color='white')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 6. Feature Importance (Coefficients)
Let's extract the logistic regression coefficients to understand what drives failures.

In [ ]:
# Extract feature names after preprocessing
cat_encoder = model.named_steps['preprocessor'].named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(categorical_features)
feature_names = numeric_features + list(cat_features)

# Extract coefficients
coefs = model.named_steps['classifier'].coef_[0]

# Create a DataFrame
importance_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
importance_df['Odds_Ratio'] = np.exp(importance_df['Coefficient'])
importance_df = importance_df.sort_values(by='Coefficient', ascending=False)

print("\n--- Feature Importance (Top factors increasing failure probability) ---")
print(importance_df.head(10))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=importance_df, palette='coolwarm')
plt.title('Logistic Regression Coefficients (Impact on Failure)', color='white')
plt.xlabel('Coefficient Value (Positive = Higher Failure Risk)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Conclusion
- **High Coefficients:** Specific categories (like Electronics) and devices (Mobile) have high positive coefficients, meaning they increase the log-odds of a transaction failing.
- **Night Time:** Transactions during the night have a strong positive coefficient.
- **Amount:** A positive coefficient for amount implies higher values are slightly more likely to fail.

*This model can be deployed to provide real-time risk scoring for transactions, potentially routing high-risk ones through more resilient payment gateways.*